In [ ]:
# 需要手动筛选的
淘天集团|https://talent.taotian.com/off-campus/position-list?lang=zh
淘宝闪购|https://talent.ele.me/off-campus/position-list?lang=zh&positionType=103
飞猪|https://career.fliggy.com/off-campus/position-list?lang=zh
阿里国际|https://aidc-jobs.alibaba.com/off-campus/position-list?lang=zh
阿里云|https://careers.aliyun.com/off-campus/position-list?lang=zh
通义实验室|https://careers-tongyi.alibaba.com/off-campus/position-list?lang=zh
钉钉|https://talent.dingtalk.com/off-campus/position-list?lang=zh
千问C端事业群|https://talent.quark.cn/off-campus/position-list?lang=zh
高德地图|https://talent.amap.com/off-campus/position-list?lang=zh
菜鸟集团|https://talent.cainiao.com/social-recruitment
虎鲸文娱集团|https://jobs.hujing-dme.com/off-campus/position-list?lang=zh
阿里健康|https://careers.alihealth.cn/off-campus/position-list?lang=zh
灵犀互娱|https://talent.lingxigames.com/off-campus/position-list?lang=zh
菜鸟驿站|https://talent-post.alibaba.com/off-campus/position-list?lang=zh
# 没有[更新日期]字段的
米哈游|https://jobs.mihoyo.com/#/position?jobName=&competencyTypes%5B0%5D=5&competencyTypes%5B1%5D=8
小红书|https://job.xiaohongshu.com/social/position?positionName=&jobTypes=om&workplaces=3100%2C4403%2C3301
字节|https://jobs.bytedance.com/experienced/position?keywords=&category=6704215882479962371%2C6704215882438019342%2C6704215908782442766%2C6704215955154667787%2C6704215961064442123%2C6704216001937934599%2C6704216057269192973%2C6704216853931100430%2C6704217437631416580%2C6704219199050352903%2C6709824273306880267%2C6850051246221429006%2C6863074795655792910%2C6704215901438216462%2C6704215901392079117%2C6704216021651163395%2C6704216386178124040%2C6704216430973290760%2C6704216870330829070%2C6704216950135851275%2C6704217388763580683&location=CT_125%2CCT_128%2CCT_52&project=&type=&job_hot_flag=&current={page_num}&limit=10&functionCategory=&tag=
# ————待办————
B站|https://jobs.bilibili.com/social/positions?code=03&type=3&onlyHotRecruit=1&page=1
鹰角|https://jobs.hypergryph.com/apply/hypergryph/26325/#/jobs?page=2&location%5B0%5D=%E4%B8%8A%E6%B5%B7%E5%B8%82&commitment%5B0%5D=%E5%85%A8%E8%81%8C&zhineng%5B0%5D=46432&pageSize=50

# 测试

In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, ElementNotInteractableException, TimeoutException
import time
import datetime
from datetime import timedelta

# ===================== 基础配置 ======================
# 驱动路径（替换为你的驱动路径，比如msedgedriver.exe的绝对路径）
DRIVER_PATH = "msedgedriver.exe"
# B站招聘目标URL（热招岗位+code=03，可根据需要修改）
TARGET_URL = "https://jobs.bilibili.com/social/positions?code=03&type=3&page=1"

# 浏览器配置（禁用自动化检测，伪装真实浏览器）
edge_options = webdriver.EdgeOptions()
edge_options.add_argument("--disable-blink-features=AutomationControlled")
edge_options.add_argument("--start-maximized")  # 最大化窗口
edge_options.add_argument("--disable-popup-blocking")
# 伪装UA
edge_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36 Edg/123.0.0.0")
# 禁用selenium自动化特征
edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
edge_options.add_experimental_option("useAutomationExtension", False)

# 初始化驱动和显式等待
driver = webdriver.Edge(options=edge_options)
driver.set_page_load_timeout(60)
wait = WebDriverWait(driver, 10)

# 存储【近两天】的岗位数据
valid_job_data = []

# ===================== 时间筛选工具函数 ======================
def parse_release_time(time_str):
    """解析B站岗位发布时间为datetime对象（兼容所有B站时间格式）"""
    now = datetime.datetime.now()
    time_str = str(time_str).strip()

    # 格式1：2026-03-30
    if '-' in time_str:
        try:
            return datetime.datetime.strptime(time_str, '%Y-%m-%d')
        except ValueError:
            return now
    # 格式2：今天
    elif '今天' in time_str:
        return now
    # 格式3：昨天
    elif '昨天' in time_str:
        return now - timedelta(days=1)
    # 格式4：3天前 / 1天前
    elif '天前' in time_str:
        days = int(''.join(filter(str.isdigit, time_str)))
        return now - timedelta(days=days)
    # 异常格式
    else:
        return now

def is_recent_two_days(parsed_time):
    """判断时间是否在【近48小时/两天内】"""
    two_days_ago = datetime.datetime.now() - timedelta(days=8)
    return parsed_time >= two_days_ago

# ===================== 核心爬取函数 ======================
def crawl_job_detail(job_card, page_num, job_idx):
    """爬取单个岗位的完整信息"""
    job_info = {
        "页码": page_num,
        "页内序号": job_idx,
        "岗位名称": "未获取",
        "工作地点": "未获取",
        "岗位类别": "未获取",
        "工作性质": "未获取",
        "发布时间": "未获取",
        "工作职责": "无",
        "任职要求": "无",
        "详情页URL": "无"
    }

    try:
        # 1. 提取列表页基础信息
        try:
            job_name_elem = job_card.find_element(By.CLASS_NAME, "item-title")
            job_info["岗位名称"] = job_name_elem.text.strip()
        except NoSuchElementException:
            print(f"⚠️ 第{page_num}页第{job_idx}个岗位：未找到岗位名称")

        # 提取地点/类别/性质/发布时间
        try:
            infotags_elem = job_card.find_element(By.CLASS_NAME, "bili-infotags")
            span_list = infotags_elem.find_elements(By.TAG_NAME, "span")
            if len(span_list) >= 1:
                job_info["工作地点"] = span_list[0].text.strip()
            if len(span_list) >= 2:
                job_info["岗位类别"] = span_list[1].text.strip()
            if len(span_list) >= 3:
                job_info["工作性质"] = span_list[2].text.strip()
            if len(span_list) >= 4:
                job_info["发布时间"] = span_list[3].text.strip().replace("发布", "").strip()
        except NoSuchElementException:
            print(f"⚠️ 第{page_num}页第{job_idx}个岗位：未找到基础信息")

        # 2. 进入详情页
        try:
            list_tab = driver.current_window_handle
            driver.execute_script("arguments[0].click();", job_card)
            time.sleep(2)

            # 切换标签页
            if len(driver.window_handles) > 1:
                detail_tab = driver.window_handles[-1]
                driver.switch_to.window(detail_tab)
                job_info["详情页URL"] = driver.current_url

                # 提取职责要求
                try:
                    sub_title_elem = wait.until(
                        EC.presence_of_element_located((By.XPATH, "//p[@class='position-sub-title' and text()='职位描述']"))
                    )
                    desc_elem = sub_title_elem.find_element(By.XPATH, "./following-sibling::p[@class='position-desc'][1]")
                    desc_text = desc_elem.text.strip()

                    # 拆分内容
                    if "工作职责:" in desc_text:
                        resp_part = desc_text.split("工作职责:")[1]
                        job_info["工作职责"] = resp_part.split("工作要求:")[0].strip().replace("\n", "；") if "工作要求:" in resp_part else resp_part.strip().replace("\n", "；")
                    if "工作要求:" in desc_text:
                        job_info["任职要求"] = desc_text.split("工作要求:")[1].strip().replace("\n", "；")

                except NoSuchElementException:
                    print(f"⚠️ 第{page_num}页第{job_idx}个岗位：未找到职责要求")

                driver.close()
                driver.switch_to.window(list_tab)
            else:
                driver.back()
                time.sleep(2)
        except Exception as e:
            print(f"⚠️ 进入详情页失败：{str(e)}")

        print(f"✅ 第{page_num}页第{job_idx}个岗位爬取完成：{job_info['岗位名称']}")
        return job_info

    except Exception as e:
        print(f"❌ 岗位爬取异常：{str(e)}")
        return job_info

def crawl_single_page(page_num):
    """
    爬取单页岗位 + 时间筛选
    返回值：True=当前页全符合，继续下一页；False=遇到旧岗位，终止全部爬取
    """
    try:
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "space")))
        job_cards = driver.find_elements(By.CLASS_NAME, "bili-item-card")
        print(f"\n📄 第{page_num}页共找到 {len(job_cards)} 个岗位")

        if not job_cards:
            return False

        # 逐岗位处理（时间倒序，遇到旧岗位直接终止）
        for idx, card in enumerate(job_cards, 1):
            job_info = crawl_job_detail(card, page_num, idx)
            release_time = parse_release_time(job_info["发布时间"])

            # ✅ 核心筛选：近两天则保留，否则直接终止所有爬取
            if is_recent_two_days(release_time):
                valid_job_data.append(job_info)
            else:
                print(f"\n🚫 第{page_num}页第{idx}个岗位发布时间超出两天，终止爬取！")
                return False

            time.sleep(1)

        # 当前页所有岗位都符合条件
        return True

    except Exception as e:
        print(f"❌ 页面爬取失败：{str(e)}")
        return False

def click_next_page():
    """点击下一页"""
    try:
        next_btn = wait.until(EC.element_to_be_clickable((By.CLASS_NAME, "ant-pagination-next")))
        if "disabled" in next_btn.get_attribute("class") or not next_btn.is_enabled():
            return False
        driver.execute_script("arguments[0].click();", next_btn)
        time.sleep(3)
        return True
    except:
        return False

# ===================== 打印结果函数 ======================
def print_job_results():
    """格式化打印所有近两天的岗位信息"""
    if not valid_job_data:
        print("\n❌ 未找到近两天发布的岗位信息")
        return

    total = len(valid_job_data)
    print(f"\n" + "="*80)
    print(f"🎉 爬取完成！共获取【近两天】岗位信息 {total} 条")
    print("="*80)

    for i, job in enumerate(valid_job_data, 1):
        print(f"\n🔹 第{i}条岗位信息")
        print(f"岗位名称：{job['岗位名称']}")
        print(f"工作地点：{job['工作地点']}")
        print(f"岗位类别：{job['岗位类别']}")
        print(f"工作性质：{job['工作性质']}")
        print(f"发布时间：{job['发布时间']}")
        print(f"详情链接：{job['详情页URL']}")
        print(f"工作职责：{job['工作职责']}")
        print(f"任职要求：{job['任职要求']}")
        print("-"*80)

# ===================== 主程序 ======================
if __name__ == "__main__":
    try:
        print("🚀 开始爬取B站【近两天】招聘岗位信息...")
        driver.get(TARGET_URL)
        time.sleep(3)

        current_page = 1
        while True:
            print(f"\n========== 处理第 {current_page} 页 ==========")
            # 爬取当前页，返回False则终止
            if not crawl_single_page(current_page):
                break
            # 无下一页则终止
            if not click_next_page():
                print("\n📌 已无下一页，爬取结束")
                break
            current_page += 1

        # 打印最终结果
        print_job_results()

    except Exception as e:
        print(f"\n💥 程序异常：{str(e)}")
    finally:
        driver.quit()
        print("\n🔌 浏览器已关闭")

🚀 开始爬取B站【近两天】招聘岗位信息...

========== 处理第 1 页 ==========

📄 第1页共找到 10 个岗位
✅ 第1页第1个岗位爬取完成：内容生态资深数据分析师
✅ 第1页第2个岗位爬取完成：数据分析师（广告）
✅ 第1页第3个岗位爬取完成：资深数据分析师（经营分析）
✅ 第1页第4个岗位爬取完成：直播产品经理（礼物）
✅ 第1页第5个岗位爬取完成：资深直播资源运营
✅ 第1页第6个岗位爬取完成：直播产品经理（B端）
✅ 第1页第7个岗位爬取完成：数据分析师-BI方向
✅ 第1页第8个岗位爬取完成：资深AI产品运营（视频创作工具）

🚫 第1页第8个岗位发布时间超出两天，终止爬取！

🎉 爬取完成！共获取【近两天】岗位信息 7 条

🔹 第1条岗位信息
岗位名称：内容生态资深数据分析师
工作地点：上海
岗位类别：产品运营类
工作性质：全职
发布时间：2026-03-23
详情链接：https://jobs.bilibili.com/social/positions/23557
工作职责：1、牵头内容生态业务的分析工作，监控业务发展态势，为业务指标异常提供预警、监控、解读；；2、结合行业数据、市场趋势，为业务发展方向提供策略和建议； ；3、在内容带增长等多个业务方向进行深入分析，挖掘可优化点，跟进AB实验，提出业务优化策略、产出专题分析报告； ；4、与运营、产品配合，推进优化方案的落地执行，带来实际增长提升；
任职要求：1、在创作者/内容分析方向上有5年以上相关经验； ；2、数据敏感有框架性思维，擅长沟通和跨团队合作，能高效和技术及业务团队沟通；擅长数据挖掘、建模，能转化业务问题并进行解决；；3、统计/数学/计算机等相关专业本科及以上学历优先；；4、有内容行业相关从业经验、有团队管理经验优先；---熟练使用AI agent能力者优先
--------------------------------------------------------------------------------

🔹 第2条岗位信息
岗位名称：数据分析师（广告）
工作地点：上海
岗位类别：产品运营类
工作性质：全职
发布时间：2026-03-23
详情链接：https://jobs.bilibili.com/social/positions/26686


In [12]:
import requests
import json
from typing import List, Dict
# 新增：时间处理模块，用于筛选近两天数据
from datetime import datetime, timedelta

def get_163_jobs() -> List[Dict]:
    """
    爬取网易招聘岗位信息，筛选近两天更新的岗位，直接打印结果
    :return: 包含近两天岗位信息的列表
    """
    job_list = []
    # 完整请求头
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Referer": "https://hr.163.com/job-list.html",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "zh-CN,zh;q=0.9",
        "Content-Type": "application/json;charset=UTF-8",
        "X-Requested-With": "XMLHttpRequest",
        "Origin": "https://hr.163.com"
    }
    
    api_url = "https://hr.163.com/api/hr163/position/queryPage"
    # 仅请求第1页，pageSize=200覆盖全部数据
    post_data = {
        "currentPage": 1,
        "pageSize": 200,
        "postType": "08",
        "workType": "0",
        "cityIdList":[229, 2, 138],
        "lang": "zh"
    }

    try:
        print("正在获取网易招聘岗位数据...")
        # 发送单次POST请求
        response = requests.post(
            api_url,
            headers=headers,
            json=post_data,
            timeout=15
        )
        response.raise_for_status()
        response_json = response.json()

        # 检查请求是否成功
        if response_json.get("code") != 200:
            print(f"请求失败: {response_json.get('msg')}")
            return job_list

        data = response_json.get("data")
        if not data:
            print("无数据返回")
            return job_list

        jobs = data.get("list", [])
        if not jobs:
            print("未获取到岗位信息")
            return job_list

        # 解析所有岗位信息（新增：更新时间戳，用于筛选）
        for job in jobs:
            job_id = job.get("id", "")
            detail_url = f"https://hr.163.com/job-detail.html?id={job_id}&lang=zh" if job_id else ""
            # 新增：获取岗位更新时间戳（毫秒级）
            update_time_stamp = job.get("updateTime", 0)
            
            job_info = {
                "岗位名称": job.get("name", ""),
                "岗位地址": ",".join(job.get("workPlaceNameList", [])),
                "职位描述": job.get("description", "").replace("\n", " "),
                "职位要求": job.get("requirement", "").replace("\n", " "),
                "岗位详情页URL": detail_url,
                "更新时间戳": update_time_stamp  # 用于时间筛选
            }
            job_list.append(job_info)

        # ===================== 核心修改：筛选近两天更新的岗位 =====================
        now = datetime.now()
        two_days_ago = now - timedelta(days=2)  # 计算48小时前的时间
        filtered_jobs = []
        
        for job in job_list:
            stamp = job["更新时间戳"]
            if not stamp:
                continue
            # 毫秒级时间戳转换为标准时间
            update_time = datetime.fromtimestamp(stamp / 1000)
            # 筛选：更新时间 >= 两天前
            if update_time >= two_days_ago:
                # 格式化时间，方便查看
                job["更新时间"] = update_time.strftime("%Y-%m-%d %H:%M:%S")
                filtered_jobs.append(job)
        # ======================================================================

        print(f"\n数据筛选完成！总数据：{len(job_list)} 条，近两天更新：{len(filtered_jobs)} 条\n")
        return filtered_jobs

    except requests.exceptions.RequestException as e:
        print(f"请求异常: {e}")
    except Exception as e:
        print(f"解析异常: {e}")

    return []

# 新增：格式化打印岗位信息
def print_jobs(jobs: List[Dict]):
    if not jobs:
        print("❌ 暂无近两天更新的岗位信息！")
        return
    
    # 遍历打印每个岗位，分隔线区分，清晰易读
    for index, job in enumerate(jobs, 1):
        print("-" * 80)
        print(f"【岗位 {index}】")
        print(f"岗位名称：{job['岗位名称']}")
        print(f"岗位地址：{job['岗位地址']}")
        print(f"更新时间：{job['更新时间']}")
        print(f"职位描述：{job['职位描述']}")
        print(f"职位要求：{job['职位要求']}")
        print(f"详情链接：{job['岗位详情页URL']}")
    print("-" * 80)

if __name__ == "__main__":
    # 1. 获取并筛选近两天的岗位数据
    filtered_job_data = get_163_jobs()
    # 2. 直接打印结果
    print_jobs(filtered_job_data)

d:\Anaconda3\lib\site-packages\chardet\langbulgarianmodel.py:3588: RuntimeWarning: coroutine 'get_filtered_wangyi_jobs' was never awaited
  27: {  # 'ш'
d:\Anaconda3\lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (4.0.0)/charset_normalizer (2.0.4) doesn't match a supported version!
  warnings.warn(


正在获取网易招聘岗位数据...

数据筛选完成！总数据：119 条，近两天更新：3 条

--------------------------------------------------------------------------------
【岗位 1】
岗位名称：【平台】海外产品运营
岗位地址：广州市
更新时间：2026-03-30 15:20:01
职位描述：1、产品策略制定与执行： 负责产品的全生命周期运营，制定用户增长、留存及付费转化策略；基于用户反馈及数据分析，持续推动产品功能优化，提升用户体验与品牌口碑。 2、用户增长与生态建设：主动探索多样化的获客手段，驱动内外部渠道协同实现用户规模增长；负责全球用户社群运营，策划高质量社群活动，维护产品评分与玩家生态。 3、市场合作与活动运营：拓展行业资源，与游戏厂商等合作伙伴建立联系，推动联合营销、活动联动或功能嵌入等深度合作项目，提升产品市场渗透率。 4、机会洞察与策略预判： 保持对全球手游市场的高度敏锐，前瞻性捕捉爆款游戏上线、网络波动等市场机会点，快速输出差异化运营方案及本地化策略。 5、数据驱动与商业化探索： 建立并完善数据监控体系，通过深度分析用户行为数据驱动决策，挖掘有效的商业化增长点，持续提升运营效能。
职位要求：1、3年以上互联网工具类 App 或游戏运营经验，英语能够作为日常工作语言，具备成功的增长案例或海外项目运营背景，有跨文化协作经验者优先。 2、具备极强的探索意识和自驱动力，能快速捕捉并跟进全球爆款游戏上线等市场机会。 3、熟悉游戏玩家需求及痛点，对游戏工具类产品及相关技术原理有一定了解者优先。 4、具备扎实的数据分析能力，能够熟练运用相关工具辅助业务复盘与决策优化。 5、具备优秀的资源整合与沟通谈判能力，能独立、主动地推动跨团队及跨公司合作。 6、目标导向，抗压性强，能够适应快节奏的工作环境并高效达成项目目标。
详情链接：https://hr.163.com/job-detail.html?id=74769&lang=zh
--------------------------------------------------------------------------------
【岗位 2】
岗位名称：高级用户运营（阴阳师）
岗位地址：广州市
更新时间：2026-03-30 

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from datetime import date, timedelta
import time

def get_filtered_dji_jobs():
    # ===================== 核心配置 =====================
    # 大疆仅爬第一页
    BASE_URL = "https://we.dji.com/zh-CN/social?from=home_page&category=301_302&location=3100_4403&pageSize=100&page=1"
    # 排除关键词：硕士+工作年限
    EXCLUDE_KEYWORDS = ['硕士','3年','4年','5年','6年','7年','8年','9年','10年','三年','四年','五年']
    
    # ===================== 【你的代码风格】ARM Chromium 配置 =====================
    options = Options()
    # 必选参数（Docker+ARM 必备）
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")

    # 【关键】连接本地 Docker 中的 seleniarm 浏览器（容器间通信地址）
    driver = webdriver.Remote(
        command_executor="http://192.168.2.53:4444/wd/hub",
        options=options
    )
    driver.set_page_load_timeout(60)
    all_jobs = []

    # 获取近2天日期
    def get_valid_dates():
        today = date.today()
        yesterday = today - timedelta(days=1)
        return [today.strftime("%Y-%m-%d"), yesterday.strftime("%Y-%m-%d")]

    # 爬取岗位详情
    def crawl_job_detail(job_item):
        try:
            job_name = job_item.find_element(By.CLASS_NAME, "PositionCard_text__2BdZa").text.strip()
            keyword_text = job_item.find_element(By.CLASS_NAME, "PositionCard_keyword__FFaH5").text.strip()
            keyword_parts = [part.strip() for part in keyword_text.split("|")]
            city = keyword_parts[0]
            update_time = keyword_parts[-1]  # 直接获取日期：2026-03-18
            detail_url = job_item.find_element(By.TAG_NAME, "a").get_attribute("href")

            # 打开详情页
            main_handle = driver.current_window_handle
            driver.execute_script("window.open(arguments[0]);", detail_url)
            time.sleep(2)
            driver.switch_to.window(driver.window_handles[-1])
            time.sleep(2)

            # 提取任职要求
            requirement = ""
            subtitles = driver.find_elements(By.CLASS_NAME, "detail_subtitle__gOlwP")
            contents = driver.find_elements(By.CLASS_NAME, "detail_phases__PyEga")
            for i, sub in enumerate(subtitles):
                if "任职要求" in sub.text and i < len(contents):
                    requirement = contents[i].text.strip()

            driver.close()
            driver.switch_to.window(main_handle)
            return {
                "岗位名": job_name,
                "工作地点": city,
                "详情链接": detail_url,
                "更新时间": update_time,
                "岗位要求": requirement
            }
        except Exception:
            if len(driver.window_handles) > 1:
                driver.close()
                driver.switch_to.window(driver.window_handles[0])
            return None

    # ===================== 主爬取逻辑 =====================
    try:
        valid_dates = get_valid_dates()
        driver.get(BASE_URL)
        time.sleep(5)
        
        # 获取岗位列表
        job_items = driver.find_elements(By.CLASS_NAME, "social_position_card__epffd")
        
        for item in job_items:
            job = crawl_job_detail(item)
            if job:
                all_jobs.append(job)

        # 双重筛选
        recent_jobs = [j for j in all_jobs if j["更新时间"] in valid_dates]
        final_jobs = [j for j in recent_jobs if not any(k in j["岗位要求"] for k in EXCLUDE_KEYWORDS)]
        
        return final_jobs
    finally:
        driver.quit()

# 运行并输出结果
if __name__ == "__main__":
    result = get_filtered_dji_jobs()
    print(f"\n✅ 筛选完成，符合条件岗位：{len(result)}")
    for job in result:
        print(job)


✅ 筛选完成，符合条件岗位：3
{'岗位名': '中/高级区域销售岗（大疆行业-欧洲/东南亚/中东非/拉美）', '工作地点': '深圳市', '详情链接': 'https://we.dji.com/zh-CN/position/detail?positionId=1782728415362752512', '更新时间': '2026-03-18', '岗位要求': '1. 本科及以上学历，英语能作为工作语言，口语流利，有海外经历或小语种能力优先（日语/韩语/德语/葡语/西语/阿拉伯语），接受海外长期外派；\n2. 具备销售、渠道管理等相关经验，熟悉平台商运作模式，有海外工作经验、无人机相关工作经验优先；\n3. 具备较强的市场分析和拓展能力、渠道管理能力、执行力、沟通协调能力；\n4. 工作认真负责、善沟通协调、心态开放，具备良好的应变能力和承压能力。'}
{'岗位名': '中/高级区域销售岗（消费级产品-欧洲/北美/东南亚/澳洲/巴西/墨西哥/印度/日本/韩国）', '工作地点': '深圳市', '详情链接': 'https://we.dji.com/zh-CN/position/detail?positionId=1755151167823581184', '更新时间': '2026-01-29', '岗位要求': '1. 本科及以上学历，英语能作为工作语言，口语流利，有海外经历或小语种能力优先（日语、韩语、西语、葡语等），接受海外长期外派；\n2. 熟悉消费电子海外渠道业务模式及操作方法；\n3. 具备较强的市场分析和拓展能力、渠道管理能力；\n4. 工作认真负责、善沟通协调，具备良好的应变能力和承压能力，心态开放。'}
{'岗位名': '中/高级区域销售岗（大疆农业-欧洲/北美/日韩/拉美/巴西/东南亚/中东非）', '工作地点': '深圳市', '详情链接': 'https://we.dji.com/zh-CN/position/detail?positionId=1870066161332559872', '更新时间': '2025-08-11', '岗位要求': '1. 本科及以上学历，英语能作为工作语言，口语流利，有海外经历或小语种能力优先（日语/韩语/德语/葡语/西语/阿拉伯语），接受海外长期外派；\n2. 具备销售、渠道管理等相关经验，熟悉平